# Rule: **build_hourly_heat_demand**

This rule builds hourly heat demand time series from daily heat demand.

Water and space heating demand profiles are generated using intraday profiles from BDEW. Different profiles are applied for the residential and services sectors, as well as for weekdays and weekends. The daily heat demand is multiplied by the intraday profile to obtain the hourly heat demand time series.

In addition, a demand-side management (DSM) availability profile is produced for residential space heating. It is a weekly binary profile indicating the hours at which the heat storage (building thermal mass) must be depleted, restricting long-term storage while allowing short-term load shifting.

**Outputs**

- resources/`hourly_heat_demand_total_base_s_{clusters}.nc`
- resources/`residential_heat_dsm_profile_total_base_s_{clusters}.csv`

In [ ]:
######################################## Parameters

### Run
name = ''
prefix = ''

### Network
clusters = 5

### Spatial domain 'ES' or 'EU' (for maps domain)
spatial_domain = 'ES'

In [ ]:
##### Import packages
import xarray as xr
import pandas as pd
import numpy as np
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import os
import sys


##### Import local functions
sys.path.append(os.path.abspath(os.path.join('..')))
import functions as xp


##### Read params.yaml
params = xp.read_params('../params.yaml')


##### Ignore warnings
import warnings
warnings.filterwarnings('ignore', category=UserWarning)


##### Region files
region_tag = f'base_s_{clusters}'
gdf_regions_onshore, _ = xp.load_regions(
    params,
    prefix=prefix,
    name=name,
    region_tag=region_tag,
)

## `hourly_heat_demand_total_base_s_{clusters}.nc`

Load the dataset and show its structure.

In [ ]:
file = f'hourly_heat_demand_total_base_s_{clusters}.nc'
path = f'{params["rootpath"]}/resources/{prefix}/{name}/'

ds = xr.open_dataset(path+file)

ds

The dataset contains four heat demand variables (`residential water`, `residential space`, `services water`, `services space`), indexed by `snapshots` (hourly timestamps) and `node` (clustered buses).

#### Summary

What is the aggregated annual heat demand (TWh) per sector use and node?

In [ ]:
#################### Annual demand per sector use and node [TWh]
annual = pd.DataFrame(
    {v: ds[v].sum(dim='snapshots').to_series() / 1e6 for v in ds.data_vars}
)

annual.index.name = 'node'

styled_table = annual.style \
    .set_caption('Annual heat demand [TWh] per sector use and node.') \
    .format('{:.2f}') \
    .background_gradient(cmap='Reds')

display(styled_table)

#### Time series

How does the aggregated hourly heat demand (summed over nodes) look for each sector use over the year?

In [ ]:
#################### Parameters

### Define period
start = '2013-01-01'
end = '2013-12-31'



#################### Figure
fig_size = [12,5]
fig, ax = plt.subplots(figsize=fig_size)

for v in ds.data_vars:
    serie = ds[v].sum(dim='node').to_series().loc[start:end]
    ax.plot(serie.index, serie.values, label=v, linewidth=.6, alpha=.8)

ax.grid(True, linestyle='--', alpha=0.5)
ax.set_ylabel('MW')
ax.set_title('Hourly heat demand aggregated over nodes')
ax.legend()

How does the hourly heat demand look for a given sector use and node along a selected period?

In [ ]:
#################### Parameters

### Select sector use (uncomment one of the following)
sector_use = 'residential space'
# sector_use = 'residential water'
# sector_use = 'services space'
# sector_use = 'services water'

### Define period
start = '2013-01-15'
end = '2013-01-22'



#################### Figure
fig_size = [12,4]
fig, ax = plt.subplots(figsize=fig_size)

df = ds[sector_use].to_pandas().loc[start:end]
df.plot(ax=ax, alpha=.8, linewidth=.7)

ax.grid(True, linestyle='--', alpha=0.5)
ax.set_ylabel('MW')
ax.set_title(f'Hourly heat demand - {sector_use}')
ax.legend(title='node')

#### Maps

Plot a map showing the annual heat demand [TWh] per region, for a selected sector use (or the sum of all of them).

In [ ]:
#################### Parameters

### Select sector use (uncomment one of the following)
sector_use = 'all'
# sector_use = 'residential space'
# sector_use = 'residential water'
# sector_use = 'services space'
# sector_use = 'services water'



#################### Local params
params_local = {}
params_local['vmin'] = ''   # Leave empty for automatic value
params_local['vmax'] = ''   # Leave empty for automatic value



#################### Data
if sector_use == 'all':
    annual_by_node = sum(ds[v].sum(dim='snapshots') for v in ds.data_vars).to_series() / 1e6
else:
    annual_by_node = ds[sector_use].sum(dim='snapshots').to_series() / 1e6

annual_by_node.name = 'annual_demand'
gdf = gdf_regions_onshore.merge(annual_by_node, left_on='name', right_index=True, how='left')



#################### Figure
fig_size = [12,6]
crs = ccrs.PlateCarree()

fig, ax = plt.subplots(figsize=fig_size, subplot_kw={'projection': crs})


### Fix params_local
vmin = params_local['vmin'] if params_local['vmin'] != '' else gdf['annual_demand'].min()
vmax = params_local['vmax'] if params_local['vmax'] != '' else gdf['annual_demand'].max()


### Add map features
xp.map_add_features(ax, params['map_add_features'])


### Add annual demand at regions
gdf.plot(ax=ax, column='annual_demand',
         cmap='Reds', edgecolor='black', lw=0.2,
         vmin=vmin, vmax=vmax,
         legend=True)

total = gdf['annual_demand'].sum()
ax.set_title(f'Annual heat demand ({sector_use}) [TWh]. Total: {total:.2f} TWh')

Plot a grid of maps, one per sector use, showing the annual heat demand per region.

In [ ]:
#################### Parameters
sector_uses = list(ds.data_vars)



#################### Figure
fig_size = [14,8]
crs = ccrs.PlateCarree()

fig, axes = plt.subplots(2, 2, figsize=fig_size, subplot_kw={'projection': crs})

for ax, sector_use in zip(axes.flatten(), sector_uses):

    annual_by_node = ds[sector_use].sum(dim='snapshots').to_series() / 1e6
    annual_by_node.name = 'annual_demand'
    gdf = gdf_regions_onshore.merge(annual_by_node, left_on='name', right_index=True, how='left')

    ### Add map features
    xp.map_add_features(ax, params['map_add_features'])

    ### Add annual demand at regions
    gdf.plot(ax=ax, column='annual_demand',
             cmap='Reds', edgecolor='black', lw=0.2,
             legend=True)

    total = gdf['annual_demand'].sum()
    ax.set_title(f'{sector_use} [TWh]. Total: {total:.2f}')

plt.tight_layout()

## `residential_heat_dsm_profile_total_base_s_{clusters}.csv`

Load the file and show its structure. The file has a multi-index header (`sector use`, `node`) and a datetime index.

In [ ]:
file = f'residential_heat_dsm_profile_total_base_s_{clusters}.csv'
path = f'{params["rootpath"]}/resources/{prefix}/{name}/'

dsm = pd.read_csv(path+file, header=[0,1], index_col=0, parse_dates=True)

dsm.head()

The profile is binary: `1` means heat storage (building thermal mass) is available; `0` marks the checkpoint hours at which storage must be empty.

#### Summary

At which hours of the day does the profile impose a restriction? (the profile is the same for all nodes).

In [ ]:
### Take the first node (profile is the same for all nodes)
serie = dsm.iloc[:, 0]

### Hours with restriction
restriction_hours = sorted(serie[serie == 0].index.hour.unique().tolist())
print(f'Restriction hours (storage must be empty): {restriction_hours}')

### Number of restriction timestamps per column
n_zeros = (dsm == 0).sum()
n_zeros.name = 'restriction_timestamps'
n_zeros.to_frame()

#### Time series

How does the DSM profile look along a selected week for a given node?

In [ ]:
#################### Parameters

### Select node
node = dsm.columns.get_level_values('node')[0]

### Define period (one week)
start = '2013-01-02'
end = '2013-01-08'



#################### Figure
fig_size = [12,3]
fig, ax = plt.subplots(figsize=fig_size)

serie = dsm.xs(node, level='node', axis=1).iloc[:, 0].loc[start:end]
ax.step(serie.index, serie.values, where='post', color='tab:blue')
ax.fill_between(serie.index, 0, serie.values, step='post', alpha=0.3)

ax.set_ylim(-0.1, 1.1)
ax.set_yticks([0, 1])
ax.grid(True, linestyle='--', alpha=0.5)
ax.set_ylabel('DSM availability')
ax.set_title(f'Residential heat DSM profile - node {node}')

#### Weekly pattern

Show the weekly profile as a heatmap (day of the week x hour of the day) to visualize the restriction pattern.

In [ ]:
#################### Data
serie = dsm.iloc[:, 0]

weekly = serie.groupby([serie.index.dayofweek, serie.index.hour]).mean().unstack()
weekly.index = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
weekly.columns.name = 'hour'



#################### Figure
fig_size = [12,3]
fig, ax = plt.subplots(figsize=fig_size)

im = ax.imshow(weekly.values, aspect='auto', cmap='RdYlGn', vmin=0, vmax=1)

ax.set_xticks(np.arange(24))
ax.set_xticklabels(np.arange(24))
ax.set_yticks(np.arange(len(weekly.index)))
ax.set_yticklabels(weekly.index)

ax.set_xlabel('hour of day')
ax.set_title('Residential heat DSM profile - weekly pattern (1: available, 0: restriction)')

plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()